# Старцева Мария, Кульпина Анастасия 

Применить на реальных данных о ходе торгов российскими акциями теорию эффективных портфелей для решения практической задачи о рассредоточении капитала по различным видам ценных бумаг в условиях неопределённости. 

1. Собрать данные о ежедневных ценах закрытия 30 российских акций за период с 01.01.2015 по 31.12.2025. 

выбранные акции (по тикерам) — GCHE (ПАО «Группа Черкизово»), APTK (ПАО «Аптечная сеть 36,6»), KUZB (ПАО Банк «Кузнецкий»), MGNT (ПАО «Магнит»), MOEX (ПАО «Московская биржа»), MTLR (ПАО «Мечел»), NLMK (ПАО «НЛМК»), FEES (ПАО «ФСК – Россети»), RASP (ПАО «Распадская»), SBER (ПАО Сбербанк), PLZL (ПАО «Полюс»), PIKK (ПАО «ПИК‑Специализированный застройщик»), TATN (ПАО «Татнефть»), UKUZ (ПАО «Южный Кузбасс»), VTBR (ПАО Банк ВТБ), PHOR (ПАО «ФосАгро»), SNGS (ПАО «Сургутнефтегаз»), MVID (ПАО «М.Видео»), AFLT (ПАО «Аэрофлот – российские авиалинии»), UTAR (ПАО «Авиакомпания ЮТэйр»), NMTP (ПАО «Новороссийский морской торговый порт»), IRKT (ПАО «Иркутскэнерго»), TRMK (ПАО «Трубная металлургическая компания»), ALRS (ПАО «АЛРОСА»), CHMF (ПАО «Северсталь»), BRZL (ПАО «Бурятзолото»), BANE (ПАО АНК «Башнефть»), MTSS (ПАО «МТС»), NVTK (ПАО «Новатэк»).

In [64]:
import requests
import pandas as pd

In [65]:
BASE_URL = "https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/{secid}.json"

def load_history_one(secid, start="2015-01-01", end="2025-12-31"):
    """История дневных торгов для одного тикера с ISS MOEX (без apimoex)."""
    all_rows = []
    start_str = start.replace("-", "")
    end_str = end.replace("-", "")
    start_pos = 0

    with requests.Session() as session:
        while True:
            params = {
                "from": start_str,
                "till": end_str,
                "start": start_pos,
            }
            url = BASE_URL.format(secid=secid)
            resp = session.get(url, params=params, timeout=15)
            resp.raise_for_status()
            data = resp.json()

            # Таблица 'history'
            table = data.get("history", {})
            columns = table.get("columns", [])
            rows = table.get("data", [])

            if not rows:
                break

            df = pd.DataFrame(rows, columns=columns)

            # Берём только дату и цену закрытия
            df = df[["TRADEDATE", "CLOSE"]].copy()
            df["SECID"] = secid
            all_rows.append(df)

            # пагинация ISS: если вернули меньше 100 строк – это конец
            if len(rows) < 100:
                break
            start_pos += 100

    if not all_rows:
        return pd.DataFrame(columns=["TRADEDATE", "CLOSE", "SECID"])

    out = pd.concat(all_rows, ignore_index=True)
    return out

In [66]:
tickers = [
    "GCHE", "APTK", "KUZB", "MGNT", "MOEX", "MTLR", "NLMK", "FEES",
    "RASP", "SBER", "PLZL", "PIKK", "TATN", "UKUZ", "VTBR", "PHOR",
    "SNGS", "MVID", "AFLT", "UTAR", "NMTP", "IRKT", "TRMK",
    "ALRS", "CHMF", "BRZL", "BANE", "MTSS", "NVTK",
]
tickers = sorted(set(tickers))

start = "2015-01-01"
end   = "2025-12-31"
min_obs = 2600

all_df = []
rows_per_ticker = {}  # сюда будем записывать число наблюдений

for secid in tickers:
    print("Загружаю", secid)
    df = load_history_one(secid, start=start, end=end)
    if df.empty:
        print("  нет данных")
        rows_per_ticker[secid] = 0
        continue

    # оставим только даты в нужных границах, на всякий случай
    df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
    df = df[(df["TRADEDATE"] >= start) & (df["TRADEDATE"] <= end)]

    # считаем непустые close
    n_non_na = df["CLOSE"].notna().sum()
    rows_per_ticker[secid] = n_non_na

    if n_non_na < min_obs:
        print(f"  мало данных: {n_non_na} < {min_obs}")
        # всё равно можем сохранить, если хочешь анализировать позже
        all_df.append(df)
    else:
        all_df.append(df)

# объединяем всё в один DataFrame
hist = pd.concat(all_df, ignore_index=True)

Загружаю AFLT
Загружаю ALRS
Загружаю APTK
Загружаю BANE
Загружаю BRZL
Загружаю CHMF
Загружаю FEES
Загружаю GCHE
Загружаю IRKT
Загружаю KUZB
Загружаю MGNT
Загружаю MOEX
Загружаю MTLR
Загружаю MTSS
Загружаю MVID
Загружаю NLMK
Загружаю NMTP
Загружаю NVTK
Загружаю PHOR
Загружаю PIKK
Загружаю PLZL
Загружаю RASP
Загружаю SBER
Загружаю SNGS
Загружаю TATN
Загружаю TRMK
Загружаю UKUZ
Загружаю UTAR
Загружаю VTBR


In [67]:
bad = {t: n for t, n in rows_per_ticker.items() if n < min_obs}
print("\nТикеры с недостаточным числом наблюдений (< 2600):")
for t, n in bad.items():
    print(f"{t}: {n}")

print("\nТикеры с достаточным числом наблюдений (>= 2700):")
good = [t for t, n in rows_per_ticker.items() if n >= min_obs]
print(good)


Тикеры с недостаточным числом наблюдений (< 2600):

Тикеры с достаточным числом наблюдений (>= 2700):
['AFLT', 'ALRS', 'APTK', 'BANE', 'BRZL', 'CHMF', 'FEES', 'GCHE', 'IRKT', 'KUZB', 'MGNT', 'MOEX', 'MTLR', 'MTSS', 'MVID', 'NLMK', 'NMTP', 'NVTK', 'PHOR', 'PIKK', 'PLZL', 'RASP', 'SBER', 'SNGS', 'TATN', 'TRMK', 'UKUZ', 'UTAR', 'VTBR']


In [68]:
hist = pd.concat(all_df, ignore_index=True)
hist.rename(columns={"TRADEDATE": "date", "CLOSE": "close"}, inplace=True)
hist["date"] = pd.to_datetime(hist["date"])
hist = hist.sort_values(["SECID", "date"])

hist.to_csv("moex_close_2015_2025_no_apimoex.csv", index=False)
print("Готово, строк:", len(hist))

Готово, строк: 80649


In [69]:
check = pd.read_csv("moex_close_2015_2025_no_apimoex.csv")
check.head(100)

,date,close,SECID
0,2015-01-05,33.21,AFLT
1,2015-01-06,33.07,AFLT
2,2015-01-08,35.17,AFLT
3,2015-01-09,34.00,AFLT
4,2015-01-12,34.45,AFLT
...,...,...,...
95,2015-05-26,40.23,AFLT
96,2015-05-27,40.40,AFLT
97,2015-05-28,40.75,AFLT
98,2015-05-29,41.10,AFLT


In [70]:
# считаем число наблюдений по каждому SECID и размечаем строки
counts = check.groupby('SECID')['SECID'].transform('size')
mask = counts < 2781

# строки, где по тикеру меньше 2781 наблюдения
bad_rows = check[mask]

print(bad_rows['SECID'].unique())
print(len(bad_rows))

[]
0


In [71]:
check.tail()

,date,close,SECID
80644,2025-12-24,71.60,VTBR
80645,2025-12-25,71.40,VTBR
80646,2025-12-26,72.92,VTBR
80647,2025-12-29,72.17,VTBR
80648,2025-12-30,72.22,VTBR


In [72]:
check['close'].isna().sum()

np.int64(913)

Проверка пропусков и корреляции (в идеале p<0.5)

In [73]:
prices = (
    hist
    .pivot(index="date", columns="SECID", values="close")
    .sort_index()
)

In [74]:
corr = prices.corr()

если p<1, то диверсификация принесет выгоду, чем ближе к -1, тем она выгоднее

In [75]:
import pandas as pd
import numpy as np

# prices: строки = даты, столбцы = тикеры
corr = prices.corr()

# убираем диагональ
np.fill_diagonal(corr.values, np.nan)

# сбросим имена осей, чтобы не конфликтовали
corr.index.name = None
corr.columns.name = None

corr_pairs = (
    corr
    .stack()
    .reset_index()
    .rename(columns={"level_0": "asset_1", "level_1": "asset_2", 0: "corr"})
)

# убрать дубли типа (A,B) и (B,A)
corr_pairs["pair"] = corr_pairs.apply(
    lambda x: tuple(sorted([x["asset_1"], x["asset_2"]])), axis=1
)
corr_pairs = corr_pairs.drop_duplicates(subset="pair").drop(columns="pair")

# фильтр по |corr| > 0.5
high_corr = corr_pairs[(corr_pairs["corr"]) > 0.8].sort_values("corr", ascending=False)
n_pairs = len(high_corr)
print(n_pairs)

13


In [76]:
print(30*29)

870
